# Complete Python SDK Reference

```{important}
This notebook is in the process of being migrated to Vast Data Platform Field Docs.  It will probably not run yet.
```

## Credits 

Simon Golan

## What is a Vast DB SDK

**`Vast DB SDK`** is a Python based API designed for interacting with VastDB, enabling operations such as schema and table management, data querying, and transaction handling.  
Key libraries used in this API include requests for HTTP requests, pyarrow for handling Apache Arrow data formats, and flatbuffers for efficient serialization of data structures.


## Getting started

### Requirements

If you are running this tutorial on an AWS Loopback Cluster, these steps have already been performed for you.

- Linux / Windows / MacOS client server connected on data access with VAST cluster
- Virtual IP pool configured with DNS service
  - [Network Access through Virtual IPs](https://support.vastdata.com/hc/en-us/articles/11910700861980-Configuring-Network-Access)
- VIP DNS name - for reaching all availble Cnodes in the API requests
  - [Configuring the VAST Cluster DNS Service](https://support.vastdata.com/hc/en-us/articles/11910668308252-Configuring-the-VAST-Cluster-DNS-Service)
- Python 3.7 or above
- S3 User Access & Secret Keys on VAST cluster
  - [Managing S3 User Access ](https://support.vastdata.com/hc/en-us/articles/9859972743580#UUID-6a15026a-d0bd-ebe6-e281-4b980674cecc_section-idm4577174596542432775810296988)
- Tabular Identity Policy with the proper permissions 
  - [Creating Identity Policies](https://support.vastdata.com/hc/en-us/articles/9859958983708#UUID-c381c613-81bd-3c09-69c4-ba4dcd8e8d6d)
- Bucket for Database tables  
  - [Database](https://support.vastdata.com/hc/en-us/articles/11910669299356#UUID-af9c563e-30e0-a1ae-bc67-2e14ed413176_section-idm4556557050436833513489399788https://support.vastdata.com/hc/en-us/articles/11910669299356#UUID-af9c563e-30e0-a1ae-bc67-2e14ed413176_section-idm4556557050436833513489399788)- Bucket


## Install the Vast DB SDK

In [1]:
!pip install vastdb | tail -5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.8 MB/s eta 0:00:00


## Import the Vast DB SDK

Before doing anything else, we need to import the vastdb api library.

Note that we also import annotations.  This mode makes Python's behavior more strict, including raising errors for some cases where variables are accidentally undefined.

In [2]:
import vastdb
from __future__ import annotations  # Enable stricter type checking

## Creating the initial session:

We read the connection details into variables in the code below.

In [3]:
import os

ENDPOINT = os.environ['ENDPOINT']
DATABASE_NAME = os.environ['DATABASE_NAME']
DATABASE_SCHEMA = os.environ['DATABASE_SCHEMA']
ACCESS_KEY = os.environ['ACCESS_KEY']
SECRET_KEY = os.environ['SECRET_KEY']

What are these settings and where do I find them?

- ENDPOINT: 
- DATABASE_OWNER: 
- ...

In [4]:
# override the schema for this tutorial
DATABASE_SCHEMA='python-sdk-schema'

## Connect to the Vast DB

Let's create a session to the vastdb

In [5]:
import pyarrow as pa
import vastdb
import os

session = vastdb.connect(
    endpoint=ENDPOINT,
    access=ACCESS_KEY,
    secret=SECRET_KEY)

If the connection was successful, we should be able to retrieve the Vast Cluster version

In [6]:
print("Vast Cluster version: ", session.api.vast_version)

Vast Cluster version:  (5, 1, 0, 131)


## Clean previous sdk schema and tables
If this is your first time you want to go through the tutorial then please skip this section.¶

However, if you have been through this tutorial before, please make sure you execute the following commands to start from a clean state, otherwise you will encounter errors. It is also important to execute each of the notebook cells in sequence to avoid any error messages:

In [7]:
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    try:
        schema = bucket.schema(name=DATABASE_SCHEMA)
        for table in schema.tables():
            print(f"Dropping table {table.name}")
            table.drop()
        print(f"Dropping schema {schema.name}")
        schema.drop()
        print(f"Creating schema {schema.name}")
        bucket.create_schema(DATABASE_SCHEMA, fail_if_exists=False)
    except:
        pass

Dropping table pythonsdk
Dropping table pythonsdkcitizen
Dropping table flights_cnode_importer
Dropping table flights
Dropping schema python-sdk-schema
Creating schema python-sdk-schema


In [8]:
# create a function to List tables

def print_tables(database_name=DATABASE_NAME, schema_name=DATABASE_SCHEMA):
    print(f"Listing tables in: database='{database_name}' schema='{schema_name}'")
    with session.transaction() as tx:
        schema = tx.bucket(database_name).schema(name=schema_name, fail_if_missing=False)
        if not schema:
            print(f">>> Schema {schema_name} not found.")
            return
            
        if not schema.tables():
            print(">>> No tables found.")
        for table in schema.tables():
            print(f">>> Table: '{table.name}'")

print_tables(DATABASE_NAME, DATABASE_SCHEMA)

Listing tables in: database='demo-database' schema='python-sdk-schema'
>>> No tables found.


## Data Interchange
VAST-DB uses the Apache Arrow data-interchange format for in-memory representation and high speed processing. The adoption of Arrow allows VAST to interact with modern analytics tools, minimizing or eliminating the need to serialize/deserialize data structures during execution. This interoperability makes VAST-DB capable of integrating with other software in service of high-performance workloads.

## Schema Management

### `create_schema`
- **Usage**: Create a new schema (a container of tables) in a bucket..
- **Parameters**:
  - `name` (str): Name of the schema to create.
  - `fail_if_exists` (bool, optional, default=True): If True, fail if the schema name already exists in the bucket


In [9]:
# Note that we use the fail_if_exists=False so that the command doesn't
# return an error if the schema already exists
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    schema = bucket.create_schema(DATABASE_SCHEMA, fail_if_exists=False)
    print(f"Schema: {schema.name}")

Schema: python-sdk-schema


### `schemas`
- **Usage**: List all schemas in a bucket.
- **Parameters**:
  - `batch_size` (int, optional): Maximum number of schemas to retrieve (default is `1000`).

In [10]:
# Print all schemas in the Database
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    for schema in bucket.schemas():
        print(schema)

Schema(name='demo-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000003)))
Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000003)))


### `schema`
- **Usage**: Get a specific schema (a container of tables) under this schema.
- **Parameters**:
  - `name` (str): Name of the schema to get.
  - `fail_if_missing` (bool, optional, default=True): If True, fail with an exception if the schema doesn't exist, else return None


In [11]:
schema_name='python-sdk1'

with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    try:
        # here we select the schema we are interested in
        schema = bucket.schema(name=DATABASE_SCHEMA)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000004)))


### `rename`
- **Usage**: Rename this schema.
- **Parameters**:
  - `new_name` (str): New name for the schema.

In [12]:
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    schema = bucket.create_schema('TEMPORARY_SCHEMA', fail_if_exists=False)
    print(f"Schema: {schema.name}")

Schema: TEMPORARY_SCHEMA


In [13]:
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    try:
        schema = bucket.schema(name='TEMPORARY_SCHEMA', fail_if_missing=False)
    except Exception as e:
        print("Schema doesn't exist:", e)

    # if the schema exists, we can rename it
    if schema:
        try:
            schema.rename(new_name='TEMPORARY_SCHEMA_RENAMED')
            print("Schema renamed")
        except Exception as e:
            print("Unable to rename schema:", e)

Schema renamed


In [14]:
# verify the schema exists
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    schema = bucket.schema(name='TEMPORARY_SCHEMA_RENAMED')
    print(schema)

Schema(name='TEMPORARY_SCHEMA_RENAMED', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000007)))


### `drop`
- **Usage**: Delete a schema in a bucket.
- **Parameters**:
  - No parameters

In [15]:
# let's drop the schma we created in the previous step
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)

    # first retrieve the schema
    try:
        schema = bucket.schema(name='TEMPORARY_SCHEMA_RENAMED', fail_if_missing=False)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

    # then drop it
    if schema:
        try:
            schema.drop()
            print("Schema dropped")
        except Exception as e:
            print("Unable to drop schema:", e)

Schema(name='TEMPORARY_SCHEMA_RENAMED', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000008)))
Schema dropped


In [16]:
# verify the schema names

with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)
    for schema in bucket.schemas():
        print(schema)

Schema(name='demo-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000009)))
Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000009)))


## Table Management

Creation
The creation of any high-level entity (database, table, column) is a metadata operation. There is no pre-allocation of space for data. Top level objects in the element store are updated and any new data associated with them is ready to be linked when written.Creation
The creation of any high-level entity (database, table, column) is a metadata operation. There is no pre-allocation of space for data. Top level objects in the element store are updated and any new data associated with them is ready to be linked when written.


VAST DB provides support for the following data types: 

![image](./img/VAST-DB-Types.png)

### `create_table`
- **Usage**: Create a new table in a specified schema.
- **Parameters**:
  - `table_name` (str): Name of the table to create.
  - `columns` (pyarrow.Schema): Arrow Schema object ([pyarrow.Schema documentation](https://arrow.apache.org/docs/python/generated/pyarrow.Schema.html#pyarrow.Schema))
  - `fail_if_missing`  (bool, opt, default=True) If True, fail with an exception if the schema doesn't exist, else return None
  - `use_external_row_ids_allocation=False` (bool, opt, default=False) TBC

In [17]:
import pyarrow as pa
from vastdb.errors import TableExists

TABLE_NAME='pythonsdk'

# Table schemas (don't confuse with database schema) are created using 
# PyArrow (pa)
ARROW_SCHEMA = pa.schema([('column1', pa.int32()), ('column2', pa.string())])


with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)

    # first retrieve the schema
    try:
        schema = bucket.schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

    if schema:
        try:
            table = schema.create_table(table_name=TABLE_NAME, columns=ARROW_SCHEMA)
            print(f"Table created: {table.name}")
        except TableExists as e:
            print("Couldn't create table because it already exists:", e)
        except Exception as e:
            print("Couldn't create table:", e)

Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000000a)))
Table created: pythonsdk


In [18]:
print_tables(database_name=DATABASE_NAME, schema_name=DATABASE_SCHEMA)

Listing tables in: database='demo-database' schema='python-sdk-schema'
>>> Table: 'pythonsdk'


### `tables`
- **Usage**: List all tables in a schema.
- **Parameters**:
  - `table_name` (str, optional, default=None): Only return tables matching the exact table_name.

In [19]:
TABLE_NAME='pythonsdk'

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.tables(table_name=TABLE_NAME)
    print(table)

[Table(name='pythonsdk', schema=Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000000c))), handle=15821909330075796601, stats=TableStats(num_rows=0, size_in_bytes=0, is_external_rowid_alloc=False, endpoints=()), _imports_table=False)]


In [20]:
# what happens if the table doesn't exist?

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.tables(table_name='non-existing-table')
    print(table)

[]


In [21]:
# For comparison let's list all the tables using our utility function
# created earlier in this notebook.
print_tables(database_name=DATABASE_NAME, schema_name=DATABASE_SCHEMA)

Listing tables in: database='demo-database' schema='python-sdk-schema'
>>> Table: 'pythonsdk'


### `get_stats`
- **Usage**: Obtain statistics about a specific table.
- **Parameters**:
  - No parameters

In [22]:
TABLE_NAME='pythonsdk'

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.table(name=TABLE_NAME, fail_if_missing=False)
    if table:
        print(f"Getting table stats {table.name}")
        print(table.get_stats())

Getting table stats pythonsdk
TableStats(num_rows=0, size_in_bytes=0, is_external_rowid_alloc=False, endpoints=('http://11.0.0.2:9090',))


### `drop`
- **Usage**: Delete a table.
- **Parameters**:
  - No parameters

In [23]:
with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)

    # first retrieve the schema
    try:
        schema = bucket.schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

    if schema:
        try:
            arrow_schema = pa.schema([('column1', pa.int32()), ('column2', pa.string())])
            table = schema.create_table(table_name='TEMPORARY_TABLE', columns=arrow_schema)
            print(f"Table created: {table.name}")
        except TableExists as e:
            print("Couldn't create table because it already exists:", e)
        except Exception as e:
            print("Couldn't create table:", e)

Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000010)))
Table created: TEMPORARY_TABLE


In [24]:
TABLE_NAME='pythonsdk'

# first let's list the tables
print_tables(database_name=DATABASE_NAME, schema_name=DATABASE_SCHEMA)

Listing tables in: database='demo-database' schema='python-sdk-schema'
>>> Table: 'pythonsdk'
>>> Table: 'TEMPORARY_TABLE'


In [25]:
with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.table(name='TEMPORARY_TABLE', fail_if_missing=False)
    if table:
        print(f"Dropping table {table.name}")
        table.drop()

Dropping table TEMPORARY_TABLE


In [26]:
# Let's verify the table no longer exists.
print_tables(database_name=DATABASE_NAME, schema_name=DATABASE_SCHEMA)

Listing tables in: database='demo-database' schema='python-sdk-schema'
>>> Table: 'pythonsdk'


## Column Management

Add/Remove Column
Adding a column in VAST is a transactional metadata operation that does not result in any data updates or allocations in main storage. Since VAST-DB is a columnar data store, there is no impact on subsequent inserts or updates but there is also no provision for default values during column addition. Column removals are transactional and will operate similarly to data delete operations. The column is tombstoned and becomes immediately inaccessible. Async tasks then take over and rewrite/unlink data chunks as necessary in main storage. A column removal can imply a lot of background activity, similar to a large delete, relative to the amount of data in that column (sparsity, data size, etc).  Note that this asynchronous activity is budgeted for by the system to minimize impact.



### `columns`
- **Usage**: List all columns of a table.
- **Parameters**:
  - No parameters

In [27]:
TABLE_NAME='pythonsdk'

def print_columns(database_name, schema_name, table_name):
    with session.transaction() as tx:
        schema = tx.bucket(database_name).schema(schema_name)
        table = schema.table(name=table_name, fail_if_missing=False)
        if table:
            columns = table.columns()
            print(columns)
        else:
            print(f"Couldn't find the table {table_name}.")

In [28]:
print_columns(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME)

column1: int32
  -- field metadata --
  VAST:column_id: '64'
column2: string
  -- field metadata --
  VAST:column_id: '65'


### `add_columns`
- **Usage**: Add new columns to an existing table.
- **Parameters**:
  - `new_column` (Apache Arrow Schema): Schema of the columns to add.


In [29]:
# If the table 'pythonsdk' doesn't exist, go back and run the cell that creates it

TABLE_NAME='pythonsdk'

# verify the table exists
print_columns(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME)

column1: int32
  -- field metadata --
  VAST:column_id: '64'
column2: string
  -- field metadata --
  VAST:column_id: '65'


In [30]:
# let's create a utility method to check if a column exists
def column_exists(table, column_name):
    if table:
        try:
            # field(column_name) is a pyarrow method
            # https://arrow.apache.org/docs/python/generated/pyarrow.Schema.html#pyarrow.Schema.field
            cols = table.columns()
            cols.field(column_name)
            return True
        except KeyError:
            return False
        except Exception as e:
            raise e
    else:
        return False

In [31]:
import pyarrow as pa

NEW_COLUMN_NAME = 'new_column'
NEW_COLUMNS = pa.schema([(NEW_COLUMN_NAME, pa.int64())])

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.table(name=TABLE_NAME, fail_if_missing=False)
    if table:
        try:
            if column_exists(table, NEW_COLUMN_NAME):
                print("Skipping.  Column already exists.")
            else:
                print(f"Adding column to {table.name}")
                table.add_column(new_column=NEW_COLUMNS)
        except Exception as e:
            print("Couldn't add column - verify that it doesn't already exist")
            print(e)
    else:
        print(f"Couldn't find the table {TABLE_NAME}.")

Adding column to pythonsdk


In [32]:
print_columns(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME)

column1: int32
  -- field metadata --
  VAST:column_id: '64'
column2: string
  -- field metadata --
  VAST:column_id: '65'
new_column: int64
  -- field metadata --
  VAST:column_id: '66'


### `rename_column`
- **Usage**: Rename a column in a table.
- **Parameters**:
  - `current_column_name` (str): The name of the column.
  - `new_column_name` (str, optional): New column name (default is an empty string `""`).

#### Alter and rename the Column


In [33]:
CUR_COLUMN_NAME = 'new_column'
NEW_COLUMN_NAME = 'renamed_new_column'

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.table(name=TABLE_NAME, fail_if_missing=False)
    if table:
        try:
            if column_exists(table, NEW_COLUMN_NAME):
                print("Skipping.  Column already exists.")
            else:
                print(f"Adding column to {table.name}")
                table.rename_column(current_column_name=CUR_COLUMN_NAME, new_column_name=NEW_COLUMN_NAME)
        except Exception as e:
            print("Couldn't add column - verify that it doesn't already exist")
            print(e)
    else:
        print(f"Couldn't find the table {TABLE_NAME}.")


Adding column to pythonsdk


#### Validate that the name of the column has changed


In [34]:
print_columns(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME)

column1: int32
  -- field metadata --
  VAST:column_id: '64'
column2: string
  -- field metadata --
  VAST:column_id: '65'
renamed_new_column: int64
  -- field metadata --
  VAST:column_id: '66'


### `drop_column`
- **Usage**: Remove columns from a table.
- **Parameters**:
  - `column_to_drop` (Apache Arrow Schema): Schema of the columns to remove.


In [35]:
COLUMN_NAME = 'renamed_new_column'
COLUMN_TO_DROP = pa.schema([(COLUMN_NAME, pa.int64())])

with session.transaction() as tx:
    schema = tx.bucket(DATABASE_NAME).schema(DATABASE_SCHEMA)
    table = schema.table(name=TABLE_NAME, fail_if_missing=False)
    if table:
        try:
            if not column_exists(table, COLUMN_NAME):
                print("Skipping.  Column doesn't exists.")
            else:
                print(f"Dropping column from {table.name}")
                table.drop_column(column_to_drop=COLUMN_TO_DROP)
        except Exception as e:
            print("Couldn't add column - verify that it doesn't already exist")
            print(e)
    else:
        print(f"Couldn't find the table {TABLE_NAME}.")


Dropping column from pythonsdk


## Data Querying and Manipulation


### Query and Filter

**Filtering** 
VAST-DB Supports filtering using a complete set of predicates. These predicates can be used directly with the VAST-DB APIs or used by analytics engines that can “push down” operations to VAST-DB. 


**Filtering Mechanics**

When searching for data, VAST-DB will use a distributed tree scan operation over range blocks in the element store, comparing the metadata in the bitmap blocks and eliminating blocks/sections of the tree based on min/max values. The search efficiency is directly related to the randomness of the column data referred to by the bitmap blocks. Best case scenario would be a search over a column whose values range linearly (greatest to least or least to greatest) relative to the keyspace without any overlap in the index blocks - reads of data blocks to find matching data are largely eliminated in this case. While the worst case would be a search on data that is random relative to the key space forcing reads on large sections of column data to perform comparisons. VAST-DB 5.0 will support sorted projections which will act similarly to indices to enable “best case” (ordered) scenarios on a given column or columns.


![image](./img/VASTDB-SCHEMA.png)

If we compare it with Parquet it allows us to store the data in much smaller elements and at a very fine granular level which is 4000x smaller compared to a standard parquet file with 128MB. 

![image](./img/VASTDB-SCHEMA2.png)

The Data is distributed and sorted. We are keeping min and max stats and the count for each column. This allows us to allow for filter push downs and we are able to skip and reduce the amount of data that we have to transfer back to the query engine. 


![image](./img/VASTDB-PARQUET.png)



#### `select`

- **Usage**: Execute a data query on a specified table within a bucket and schema. This function allows for complex queries, including filters and projections, on the table data.
- **Parameters**:
  - `columns` (list[str], optional): List of column names to be included in the query. If None, all columns are included.
  - `predicate` (dict, optional): A filter to apply to the query. See [filters-and-projections](https://github.com/vast-data/vastdb_sdk/blob/main/README.md#filters-and-projections) for more details.
  - `config` (QueryConfig, optional): Configuration options for the query execution. If None, default configuration is used. See the source for [vastdb.table.QueryConfig](https://github.com/vast-data/vastdb_sdk/blob/main/vastdb/table.py).
  - `internal_row_id` (bool, optional): If True, includes the internal row ID in the query.

- **Returns**:
  - `pa.RecordBatchReader`: A reader object that can be iterated to retrieve the query results as record batches.
 
- **Raises**:
  - `errors.TooLargeRequest`: If the serialized query data request size exceeds the maximum allowed size.
  - `StoppedException`: If the stop event is set during the execution of the query.

### Create a table with Citizen information


#### Specify the Citizen Table


In [36]:
## Define the Table name for the citizen table

TABLE_NAME='pythonsdkcitizen'

#### Define the Citizen Table


In [37]:
import pyarrow as pa
from vastdb.errors import TableExists

ARROW_SCHEMA = pa.schema([
        ('Citizen_Age', pa.int64()),
        ('Citizen_Name', pa.string()),
        ('Citizen_experience', pa.float64()),
        ('Is_married', pa.bool_()),
    ])


with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.create_table(table_name=TABLE_NAME, columns=ARROW_SCHEMA)
                print(f"Table created: {table.name}")
            except TableExists as e:
                print("Couldn't create table because it already exists:", e)
            except Exception as e:
                print("Couldn't create table:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Table created: pythonsdkcitizen


#### Insert rows into the Citizen Table


In [38]:
ROWS = { 
    'Citizen_Name': ['Alice', 'Bob', 'Koko', 'Menny'],
    'Citizen_Age': [45, 38, 27, 51],
    'Citizen_experience': [25.5, 17.9, 5.3, 28.2],
    'Is_married': [True, False, False, True]
}
PA_RECORD_BATCH = pa.RecordBatch.from_pydict(ROWS)


with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                table.insert(PA_RECORD_BATCH)
                print("Data inserted.")
            except Exception as e:
                print("Couldn't insert data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Data inserted.


This uses the [RecordBatch.from_pydict](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html#pyarrow.RecordBatch.from_pydict) method.  See also:

- [RecordBatch.from_arrays](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html#pyarrow.RecordBatch.from_arrays)
- [RecordBatch.from_pandas](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html#pyarrow.RecordBatch.from_pandas)
- [RecordBatch.from_pylist](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html#pyarrow.RecordBatch.from_pylist)
- [RecordBatch.from_struct_array](https://arrow.apache.org/docs/python/generated/pyarrow.RecordBatch.html#pyarrow.RecordBatch.from_struct_array)

### Query the table and list all rows for the Citizen Table


In [39]:
with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select()
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True


This uses the [Table.to_pandas](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pandas) method.  See also:

- [Table.to_batches](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_batches)
- [Table.to_pydict](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pydict)
- [Table.to_pylist](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pylist)
- [Table.to_reader](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_reader)
- [Table.to_string](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_string)
- [Table.to_struct_array](https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_struct_array)

### (Filter) Predicates in VastdbApi
- You can use these predicates in the VastdbApi API:  

  - **`eq`**: Equal -> `'column_name': ['eq value']`
  - **`ge`**: Greater Than or Equal -> `'column_name': ['ge value']`
  - **`gt`**: Greater Than -> `'column_name': ['gt value']`
  - **`lt`**: Less Than -> `'column_name': ['lt value']`
  - **`le`**: Less Than or Equal -> `'column_name': ['le value']`
  - **`is_null`**: Checks for null values -> `'column_name': ['is_null']`
  - **`is_not_null`**: Checks for non-null values -> `'column_name': ['is_not_null']`
**Note:**
  - If using pandas dataframe, pandas predicates can also be used. (Perfomance might be reflected because it's not API-native filters)
    - **Example** [Query a table using Pandas predicates](#query-a-table-using-pandas-predicates)
    
      **[See more advanced examples on how to query data](#advanced-examples)**

### Filter Columns
We want to filter and project only the following columns:
- Citizen_Age
- Citizen_Name
- Citizen_experience

In [40]:
COLUMNS=['Citizen_Age', 'Citizen_Name', 'Citizen_experience']

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select(columns=COLUMNS)
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience
0,45,Alice,25.5
1,38,Bob,17.9
2,27,Koko,5.3
3,51,Menny,28.2


### Filter data and columns

**(Citizen_Age > 37) AND (Citizen_experience <= 40) limit 10**


In [41]:
from ibis import _

PREDICATE = (_.Citizen_Age > 37) & (_.Citizen_experience <= 40)
COLUMNS = ['Citizen_Age', 'Citizen_Name', 'Citizen_experience']

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select(columns=COLUMNS, predicate=PREDICATE)
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience
0,45,Alice,25.5
1,38,Bob,17.9
2,51,Menny,28.2


**(Citizen_Age = 45) OR (Citizen_Age = 51)**


In [42]:
from ibis import _

PREDICATE = (_.Citizen_Age.isin([45,51]))
COLUMNS = ['Citizen_Age', 'Citizen_Name', 'Citizen_experience']

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select(columns=COLUMNS, predicate=PREDICATE)
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience
0,45,Alice,25.5
1,51,Menny,28.2


**(Citizen_experience <= 28) AND (Citizen_Age between 30 to 60)**


In [43]:
from ibis import _

PREDICATE = (_.Citizen_experience >= 28) & (_.Citizen_Age >= 30) & (_.Citizen_Age <= 60)
COLUMNS = ['Citizen_Age', 'Citizen_Name', 'Citizen_experience']

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select(columns=COLUMNS, predicate=PREDICATE)
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience
0,51,Menny,28.2


### Query a table using Pandas predicates

This performs client side querying.


In [44]:
with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select()
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                # pandas client side querying
                display(df[(df['Citizen_Age'] == 38) & (df['Citizen_experience'] > 2)].head(10))
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
1,38,Bob,17.9,False


## Data Manipulation

CRUD operations in VAST-DB are designed to be low-latency to the user while improving the overall performance of the system by implementing background operations in such a way that unnecessary I/O is minimized or eliminated.




### `insert`

Row inserts into VAST-DB first occur into storage-class memory as discrete operations. While some combining of rows may occur when inserts are done very close together in time, it’s easier to think of a single SQL insert command as resulting in data chunks written to SCM.

![image](./img/VAST-Insert.png)

This insert occurs according to isolation rules: clients that are part of the insert transaction are exposed to the new data immediately. Clients that begin a transaction after the insert transaction is committed are then exposed to the data. As inserts and other ingest transactions occur, asynchronous tasks will trigger, combining all of these discrete objects in SCM, and writing them to read-intensive NVMe along with updates to the associated metadata structures upstream of it.

Row inserts into VAST-DB first occur into storage-class memory as discrete operations. While some combining of rows may occur when inserts are done very close together in time, it’s easier to think of a single SQL insert command as resulting in data chunks written to SCM.

- **Usage**: Inserts a RecordBatch into this table.
- **Parameters**:
  - `rows` (pa.RecordBatch): The record batch to be inserted into the table.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
  - `errors.TooWideRow`: If the row to be inserted is too wide.
- **Returns**:
  - `pa.ChunkedArray`: An array of internal row IDs of the inserted rows, if the feature is supported by the server.
  - `None`: If the feature of returning row IDs is not supported by the server.
- **Note**:
        If a row is too wide to be inserted, the method falls back to inserting in column batches.

#### Show table

In [45]:
def list_rows():
    print(f"Listing rows in: Database='{DATABASE_NAME}' Schema='{DATABASE_SCHEMA}' Table='{TABLE_NAME}'")
    with session.transaction() as tx:
        try:
            schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
            if schema:
                try:
                    table = schema.table(name=TABLE_NAME)
                    reader = table.select()
                    pyarrow_table = pa.Table.from_batches(reader)
                    df = pyarrow_table.to_pandas()
                    display(df)
                except Exception as e:
                    print("Couldn't select data:", e)
        except Exception as e:
            print("Schema doesn't exist:", e)

list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True


#### Insert the Record

In [46]:
ROWS = { 
    'Citizen_Name': ['Alice','Bob'], 'Citizen_Age': [25,24]
}
PA_RECORD_BATCH = pa.RecordBatch.from_pydict(ROWS)


with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                table.insert(PA_RECORD_BATCH)
                print("Data inserted.")
            except Exception as e:
                print("Couldn't insert data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Data inserted.


#### Show that the record was inserted

In [47]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True
4,25,Alice,NaN,None
5,24,Bob,NaN,None


### `update`

An update to a row results in new data written to SCM where metadata objects associated with the table are updated to link the newly written data in favor of the old (according to transaction isolation rules). Asynchronous tasks then combine all of the updates and rewrite data chunks in main storage as necessary to house updated data. 

![image](./img/VAST-Updates.png)



An additional async task (called snap delete) will unlink chunks if they are part of an existing snapshot or if they are needed inside or outside of an ongoing transaction.

- **Usage**: Updates a subset of cells in this table.
- **Parameters**:
  - `rows` (Union[pa.RecordBatch, pa.Table]): The rows to be updated. Must include a special field named `$row_id` of uint64 type.
  - `columns` (Optional[List[str]]): The subset of columns to be updated. If None, all columns are updated.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
  - `errors.MissingRowIdColumn`: If the `$row_id` field is missing from the rows.
- **Note**:
  -  This function assumes that the `$row_id` field is part of the input rows. The `$row_id` field is used to specify the row IDs to be updated.

#### list the table

In [48]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True
4,25,Alice,NaN,None
5,24,Bob,NaN,None


#### Update the Citizen_experience and Is_married status for Alice & Bob 


In [49]:
# Define the fields and their types
FIELDS = [
    ("$row_id", pa.uint64()),
    ("Citizen_experience", pa.float64()),
    ("Is_married", pa.bool_())
]

# Define the values for each field
VALUES = [
    [4, 5],  # values for "$row_id"
    [43, 28],  # values for "Citizen_experience"
    [False, True]  # values for "Is_married"
]

# Create a record batch
RECORD_BATCH = pa.record_batch(VALUES, schema=pa.schema(FIELDS))

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                
                # Update the table
                table.update(RECORD_BATCH)
                
                print("Data updated.")
            except Exception as e:
                print("Couldn't insert data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)


Data updated.


#### validate the updates for Alice & Bob 

In [50]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True
4,25,Alice,43.0,False
5,24,Bob,28.0,True


### `delete`

Row-level deletes work similarly to updates. Table metadata is updated, logically removing the row (tombstoning it).
Eventually asynchronous processes will then relocate (rewrite and unlink) data as necessary to reclaim space in main storage - taking care to account for ongoing transactions and snapshots.

![image](./img/VAST-Deletes.png)

- **Usage**: Deletes a subset of rows in this table.
- **Parameters**:
  - `rows` (Union[pa.RecordBatch, pa.Table]): The rows to be deleted. Must include a special field named `$row_id` of uint64 type.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
  - `errors.MissingRowIdColumn`: If the `$row_id` field is missing from the rows.
- **Note**:
  - This function assumes that the `$row_id` field is part of the input rows. The `$row_id` field is used to specify the row IDs to be deleted.

#### List the rows in the table 

In [51]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,38,Bob,17.9,False
2,27,Koko,5.3,False
3,51,Menny,28.2,True
4,25,Alice,43.0,False
5,24,Bob,28.0,True


#### Delete the Row for Bob


In [52]:
PREDICATE = (_.Citizen_Name == 'Bob') & (_.Is_married == True)
COLUMNS = ['Citizen_Age', 'Citizen_Name', 'Citizen_experience']

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)

                # IMPORTANT: internal_row_id=True
                reader = table.select(columns=COLUMNS, predicate=PREDICATE, internal_row_id=True)
                
                pyarrow_table = reader.read_all()
                if pyarrow_table.num_rows == 0:
                    print(f"No records found.")
                else:
                    print("Records to delete (note we ran select with `internal_row_id=True`) ...")
                    display(pyarrow_table.to_pandas())
    
                    print("Deleting rows");
                    table.delete(pyarrow_table)
            except Exception as e:
                import sys, traceback
                traceback.print_exc(file=sys.stdout)
                print("Exception encountered:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)


Records to delete (note we ran select with `internal_row_id=True`) ...


,Citizen_Age,Citizen_Name,Citizen_experience,$row_id
0,38,Bob,17.9,1


Deleting rows


In [53]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,27,Koko,5.3,False
2,51,Menny,28.2,True
3,25,Alice,43.0,False
4,24,Bob,28.0,True


#### List the rows in the table 

## Semi-Sorted Projections Management
Introduced with VAST Version 5.0


## What is a projection?

A projection is 
a subset of a database table’s columns 
that are copied into a separate table and 
sorted into chunks for accelerated queries  

When the source table is queried for columns that have been projected, the query will access the projection for better performance.

A background process updates the projection, and if no projected value is available, the source table will be used.

![image](./img/Semi-sortedProjections.png)


## How do projections work

1. Create a projection with a subset of columns, choosing which column(s) to sort by
2. A background process creates the projection table and adds pointers to the source table and the projection table 
- The projection is sharded and sorted in 8*216 = 524,288 row chunks
- The query engine see the pointers and can easily jump back and forth between the two
3. Updates and deletes don’t trigger a re-sort process (not needed)
4. Queries with the projected columns will use the projection table when possible, and if the projection doesn’t include a row, will use the source table instead.
5. End users shouldn’t need to understand the projection, they’ll just see that the query is accelerated.


### `create_projection`

- **Usage**: Creates a new semi-sorted projection on the table.
- **Parameters**:
  - `projection_name` (str): The name of the new projection.
  - `sorted_columns` (List[str]): A list of column names that should be sorted in the projection.
  - `unsorted_columns` (List[str]): A list of column names that should not be sorted in the projection.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
- **Returns**:
  - `Projection`: The newly created projection.
- **Note**:
  - A projection is a subset of a table's data, where some columns are sorted and others are not. This can be useful for optimizing certain types of queries.

In [54]:
# Confirm we are working with the correct objects

assert DATABASE_NAME == 'demo-database'
assert DATABASE_SCHEMA == 'python-sdk-schema'
assert TABLE_NAME == 'pythonsdkcitizen'

In [55]:
list_rows()

Listing rows in: Database='demo-database' Schema='python-sdk-schema' Table='pythonsdkcitizen'


,Citizen_Age,Citizen_Name,Citizen_experience,Is_married
0,45,Alice,25.5,True
1,27,Koko,5.3,False
2,51,Menny,28.2,True
3,25,Alice,43.0,False
4,24,Bob,28.0,True


In [56]:
PROJECTION_SORTED_COLUMNS = [ 'Citizen_Age' ]
PROJECTION_UNSORTED_COLUMNS = [ 'Citizen_Name', 'Is_married' ]
PROJECTION_NAME = 'demo-projection'

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                projection = table.create_projection(
                                        PROJECTION_NAME, 
                                        sorted_columns=PROJECTION_SORTED_COLUMNS, 
                                        unsorted_columns=PROJECTION_UNSORTED_COLUMNS
                                        )
                print("Projection created:", projection)
            except Exception as e:
                print("Exception encountered:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Projection created: Projection(name='demo-projection', table=Table(name='pythonsdkcitizen', schema=Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000002d))), handle=1877086195346196137, stats=TableStats(num_rows=6, size_in_bytes=169, is_external_rowid_alloc=False, endpoints=()), _imports_table=False), handle=8168954746074489682, stats=TableStats(num_rows=0, size_in_bytes=0, is_external_rowid_alloc=False, endpoints=()))


#### `projections`

- **Usage**: Lists all semi-sorted projections of this table or a specific projection if a name is provided.
- **Parameters**:
  - `projection_name` (str, optional): The name of the projection to list. If not provided, all projections are listed.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
- **Returns**:
  - `List[Projection]`: A list of Projection objects representing the projections of this table.
- **Note**:
  - A projection is a subset of a table's data, where some columns are sorted and others are not. This can be useful for optimizing certain types of queries.


In [57]:
PROJECTION_NAME = 'demo-projection'

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                projection = table.projections(PROJECTION_NAME)
                print("Projection created:", projection)
            except Exception as e:
                print("Exception encountered:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Projection created: [Projection(name='demo-projection', table=Table(name='pythonsdkcitizen', schema=Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000002e))), handle=1877086195346196137, stats=TableStats(num_rows=6, size_in_bytes=169, is_external_rowid_alloc=False, endpoints=()), _imports_table=False), handle=8168954746074489682, stats=TableStats(num_rows=0, size_in_bytes=0, is_external_rowid_alloc=False, endpoints=()))]


#### Projection Stats

The projects are synced in the backend and it can take some time until you will see the stats. 

The stats can be found in the reponse from `table.projections()`.  E.g.

```
Projection(
    name='demo-projection',
    table=Table(name='pythonsdkcitizen',
    ...
    stats=TableStats(num_rows=6, size_in_bytes=169, is_external_rowid_alloc=False, endpoints=()),
    ...
)
```

#### `rename`

- **Usage**: Renames this projection.
- **Parameters**:
  - `new_name` (str): The new name for the projection.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.
- **Note**:
  - This operation will change the name of the projection in the database. The change is reflected in the 'name' attribute of this object.

In [58]:
PROJECTION_NAME = 'demo-projection'
PROJECTION_NAME_RENAMED = 'demo-projection-renamed'

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                projection = table.projection(PROJECTION_NAME)
                print("Projection name before rename:", projection.name)

                projection.rename(PROJECTION_NAME_RENAMED)
                projection = table.projection(PROJECTION_NAME_RENAMED)
                print("Projection name after rename:", projection.name)
                
            except Exception as e:
                print("Exception encountered:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Projection name before rename: demo-projection
Projection name after rename: demo-projection-renamed


#### `drop`

- **Usage**: Drops (deletes) this table from the VAST Database.  This operation is irreversible and will permanently remove the table along with all its data and associated metadata from the database.
- **Parameters**:
  - None
- **Raises**:
  - `VastDBError`: If there is an error communicating with the VAST server or if the table cannot be dropped due to permissions issues or other server-side constraints.
- **Note**: This method will also remove any imports table associated with this table.

In [59]:
def list_projections():
    with session.transaction() as tx:
        try:
            schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
            if schema:
                try:
                    table = schema.table(name=TABLE_NAME)                    
                    projections = table.projections()
                    print(f"Found {len(projections)} projections.")
                    for p in projections:
                        print(f'>>> {p.name}')
                except Exception as e:
                    import sys, traceback
                    traceback.print_exc(file=sys.stdout)
                    print("Exception encountered:", e)
        except Exception as e:
            print("Schema doesn't exist:", e)

list_projections()

Found 1 projections.
>>> demo-projection-renamed


In [60]:
PROJECTION_NAME_RENAMED = 'demo-projection-renamed'

with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)                    
                projection = table.projection(PROJECTION_NAME_RENAMED)
                projection.drop()
            except vastdb.errors.MissingProjection:
                print("Projection not found.")
            except Exception as e:
                import sys, traceback
                traceback.print_exc(file=sys.stdout)
                print("Exception encountered:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

In [61]:
list_projections()

Found 0 projections.


## `import_file` and `import_files`

Bulk importation of data into VAST-DB, can be done with any method supported by the analytics/query engine.  
Inserts, CTAS commands (CREATE TABLE AS [QUERY]), python scripts processing data frames, etc. Tools like Spark make this easy, but this type of processing will be slow compared to the LOAD operations supported by various databases. VAST-DB includes LOAD capabilities.
 
Ingesting Serialized Data Objects (LOAD)
Parquet objects housed on VAST can be ingested directly into a VAST-DB table using an RPC call issued via the VAST-DB APIs. Issuing these calls is supported from Trino and Spark and this will usually be how these calls are made. Import process will look like this:
 
- The target table with schema is prepared in VAST-DB
- Data is placed in an S3 bucket on VAST storage
- An RPC call is made to a VAST-DB CNode directing it to the object location(s) in VAST
- The VAST-DB CNode cluster will then divide the workload and import the data directly from the file and into the prepared table. 

Currently, Parquet is the only format supported for this operation, however creating additional filters for ORC, delimited text, CSV, JSON, etc. is a simple development task (if you want it, just ask).
Bulk importation of data into VAST-DB, can be done with any method supported by the analytics/query engine. Inserts, CTAS commands (CREATE TABLE AS [QUERY]), python scripts processing data frames, etc. Tools like Spark make this easy, but this type of processing will be slow compared to the LOAD operations supported by various databases. VAST-DB includes LOAD capabilities.

---

- **Method**: table.import_file
- **Usage**: Imports a list of Parquet files into this table. The files must be on the VAST S3 server and be accessible using current credentials.
- **Args**:
  - `files_to_import` (Iterable[str]): An iterable of file paths to import.
  - `config` (Optional[ImportConfig], optional): Configuration for the import operation. Defaults to None.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.

---

- **Method**: table.import_partitioned_files
- **Usage**: Imports a list of Parquet files into this table, with each file having its own partition values. The files must be on the VAST S3 server and be accessible using current credentials. Each file must have its own partition values defined as an Arrow RecordBatch.
- **Args**:
  - `files_and_partitions` (Dict[str, pa.RecordBatch]): A dictionary mapping file paths to their corresponding partition values.
  - `config` (Optional[ImportConfig], optional): Configuration for the import operation. Defaults to None.
- **Raises**:
  - `errors.NotSupportedCommand`: If the operation is not supported on the current table.



### Import using the importer on Cnodes
The importer expects that S3 files are available in an S3 Bucket. 
For the tutorial we will upload an example airline parquet file to a VAST S3 Bucket  


In [62]:
import os

BUCKET_NAME=DATABASE_NAME
BUCKET_OWNER = os.environ['DATABASE_OWNER']

print(f"""
{ENDPOINT=} 
{DATABASE_NAME=}
{BUCKET_NAME=}
{BUCKET_OWNER=}
{ACCESS_KEY=}
""")


ENDPOINT='http://11.0.0.2:9090' 
DATABASE_NAME='demo-database'
BUCKET_NAME='demo-database'
BUCKET_OWNER='demo-owner'
ACCESS_KEY='3Q7A0MCGTXXCGFD3FQUB'
SECRET_KEY='XqpYHIzNcUuVA6+Ic73PzBsezDuBsUzM8x5Sy394'



In [63]:
# You can replace these values with your own

LOCAL_FILE_PATH = 'flights.parquet'
S3_FILE_KEY = 'pythonsdk/import/flights.parquet'

#### Import our flight data parquet file into s3 bucket

In [64]:
import os
import boto3
from botocore.exceptions import NoCredentialsError

def upload_to_s3(local_file, bucket_name, s3_file, aws_access_key_id, aws_secret_access_key, s3_endpoint):
    # Create an S3 client with custom endpoint
    s3 = boto3.client('s3', aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key, endpoint_url=s3_endpoint)

    try:
        # Upload the file
        s3.upload_file(local_file, bucket_name, s3_file)
        print(f"File {local_file} uploaded to {bucket_name}/{s3_file} successfully.")
    except FileNotFoundError:
        print(f"The file {local_file} was not found.")
    except NoCredentialsError:
        print("Credentials not available.")
        
def list_objects_in_bucket(bucket_name, aws_access_key_id, aws_secret_access_key, s3_endpoint, prefix=None):
    # Create an S3 client with custom endpoint
    s3 = boto3.client('s3', aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key, endpoint_url=s3_endpoint)

    try:
        # List objects in the bucket with optional prefix
        kwargs = {'Bucket': bucket_name}
        if prefix:
            kwargs['Prefix'] = prefix

        response = s3.list_objects_v2(**kwargs)

        if 'Contents' in response:
            print(f"Objects in bucket {bucket_name} with prefix '{prefix}':")
            for obj in response['Contents']:
                print(f">>> obj['Key']")
        else:
            print(f"No objects found in bucket {bucket_name} with prefix '{prefix}'.")
    except NoCredentialsError:
        print("Credentials not available.")


In [65]:
upload_to_s3(LOCAL_FILE_PATH, BUCKET_NAME, S3_FILE_KEY, ACCESS_KEY, SECRET_KEY, ENDPOINT)

# List objects in the bucket
list_objects_in_bucket(BUCKET_NAME, ACCESS_KEY, SECRET_KEY, ENDPOINT, S3_FILE_KEY)

File flights.parquet uploaded to demo-database/pythonsdk/import/flights.parquet successfully.
Objects in bucket demo-database with prefix 'pythonsdk/import/flights.parquet':
>>> obj['Key']


### Create Schema and use the API to start the import on the Cnodes

In [66]:
TABLE_NAME='flights_cnode_importer'

In [67]:
import pyarrow as pa
from vastdb.errors import TableExists

# create table schema before using the importer API

ARROW_SCHEMA = pa.schema([
    ('FL_DATE', pa.date32()), 
    ('DEP_DELAY', pa.int16()),
    ('ARR_DELAY', pa.int16()),
    ('AIR_TIME', pa.int16()),
    ('DISTANCE', pa.int16()),
    ('DEP_TIME', pa.float32()),
    ('ARR_TIME', pa.float32())
])

with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)

    # first retrieve the schema
    try:
        schema = bucket.schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

    if schema:
        try:
            table = schema.create_table(table_name=TABLE_NAME, columns=ARROW_SCHEMA)
            print(f"Table created: {table.name}")
        except TableExists as e:
            print("Couldn't create table because it already exists:", e)
        except Exception as e:
            print("Couldn't create table:", e)

Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000033)))
Table created: flights_cnode_importer


In [68]:
FILES_TO_IMPORT = [f'/{BUCKET_NAME}/{S3_FILE_KEY}']
print(f'Importing {FILES_TO_IMPORT}')

with session.transaction() as tx:
    bucket = tx.bucket(DATABASE_NAME)

    # first retrieve the schema
    try:
        schema = bucket.schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        table = schema.table(TABLE_NAME)
        print(schema)
    except Exception as e:
        print("Schema doesn't exist:", e)

    if table:
        try:
            table.import_files(files_to_import=FILES_TO_IMPORT)
        except Exception as e:
            import sys, traceback
            traceback.print_exc(file=sys.stdout)
            print("Couldn't import files:", e)


Importing ['/demo-database/pythonsdk/import/flights.parquet']
Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000034)))


Let's verify the import...

In [69]:
with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select()
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                print(f"Listing rows in {TABLE_NAME}")
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Listing rows in flights_cnode_importer


,FL_DATE,DEP_DELAY,ARR_DELAY,AIR_TIME,DISTANCE,DEP_TIME,ARR_TIME
0,2006-01-01,5,19,350,2475,9.083333,12.483334
1,2006-01-02,167,216,343,2475,11.783334,15.766666
2,2006-01-03,-7,-2,344,2475,8.883333,12.133333
3,2006-01-04,-5,-13,331,2475,8.916667,11.950000
4,2006-01-05,-3,-17,321,2475,8.950000,11.883333
...,...,...,...,...,...,...,...
999995,2006-01-19,5,4,244,1781,15.000000,17.350000
999996,2006-01-20,14,12,240,1781,15.150000,17.483334
999997,2006-01-21,9,12,241,1781,15.066667,17.483334
999998,2006-01-22,-2,8,242,1781,14.883333,17.416666


### Import Parquet Files using Pandas

The following example will use Pandas to load a local parquet file and insert the data into a VAST DB table

In [70]:
# Set your parameters and credentials

TABLE_NAME = 'flights'

In [71]:
# print parquet file name
print(f"{LOCAL_FILE_PATH=}")

LOCAL_FILE_PATH='flights.parquet'


In [72]:
import pandas as pd
import pyarrow.parquet as pq

DF = pd.read_parquet(LOCAL_FILE_PATH)

# Read Parquet file directly into PyArrow table
ARROW_TABLE = pq.read_table(LOCAL_FILE_PATH)
ARROW_SCHEMA = ARROW_TABLE.schema

In [73]:
def create_table(database_name, database_schema, table_name, arrow_schema):
    
    with session.transaction() as tx:
        bucket = tx.bucket(database_name)
    
        # first retrieve the schema
        try:
            schema = bucket.schema(name=database_schema, fail_if_missing=False)
            print(schema)
        except Exception as e:
            print("Schema doesn't exist:", e)
    
        if schema:
            try:
                table = schema.create_table(table_name=table_name, columns=arrow_schema)
                print(f"Table created: {table.name}")
            except TableExists as e:
                print("Couldn't create table because it already exists:", e)
            except Exception as e:
                print("Couldn't create table:", e)

In [74]:
create_table(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME, ARROW_SCHEMA)

Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x0000300000000036)))
Table created: flights


In [75]:
def insert_dataframe_to_database(database_name, database_schema, table_name, record_batch):
    with session.transaction() as tx:
        try:
            schema = tx.bucket(database_name).schema(name=database_schema, fail_if_missing=False)
            if schema:
                try:
                    table = schema.table(name=table_name)
                    table.insert(record_batch)
                    print("Data inserted.")
                except Exception as e:
                    print("Couldn't insert data:", e)
        except Exception as e:
            print("Schema doesn't exist:", e)

In [76]:
RECORD_BATCH = pa.RecordBatch.from_pandas(DF)

insert_dataframe_to_database(DATABASE_NAME, DATABASE_SCHEMA, TABLE_NAME, RECORD_BATCH)

Data inserted.


In [77]:
with session.transaction() as tx:
    try:
        schema = tx.bucket(DATABASE_NAME).schema(name=DATABASE_SCHEMA, fail_if_missing=False)
        if schema:
            try:
                table = schema.table(name=TABLE_NAME)
                reader = table.select()
                pyarrow_table = pa.Table.from_batches(reader)
                df = pyarrow_table.to_pandas()
                print(f"Listing rows in {TABLE_NAME}")
                display(df)
            except Exception as e:
                print("Couldn't select data:", e)
    except Exception as e:
        print("Schema doesn't exist:", e)

Listing rows in flights


,FL_DATE,DEP_DELAY,ARR_DELAY,AIR_TIME,DISTANCE,DEP_TIME,ARR_TIME
0,2006-01-01,5,19,350,2475,9.083333,12.483334
1,2006-01-02,167,216,343,2475,11.783334,15.766666
2,2006-01-03,-7,-2,344,2475,8.883333,12.133333
3,2006-01-04,-5,-13,331,2475,8.916667,11.950000
4,2006-01-05,-3,-17,321,2475,8.950000,11.883333
...,...,...,...,...,...,...,...
999995,2006-01-19,5,4,244,1781,15.000000,17.350000
999996,2006-01-20,14,12,240,1781,15.150000,17.483334
999997,2006-01-21,9,12,241,1781,15.066667,17.483334
999998,2006-01-22,-2,8,242,1781,14.883333,17.416666


## Snapshots Management


### `snapshots`

- **Usage**: Lists all snapshots of the current bucket. A snapshot is a read-only copy of a bucket at a certain point in time. This method returns a list of Bucket objects, each representing a snapshot.
- **Parameters**:
  - None
- **Returns**:
  - `List[Bucket]`: A list of Bucket objects representing the snapshots of the current bucket.
- **Note**:
  - The name of each snapshot is the name of the bucket followed by the snapshot name, separated by a slash.


In [78]:
with session.transaction() as tx:
    try:
        snapshots = tx.bucket(DATABASE_NAME).snapshots()
        for s in snapshots:
            print(s)
    except Exception as e:
        print("Issue retrieving snapshots:", e)

Bucket(name='demo-database/.snapshot/big_catalog_2024-06-14_08_28_15_UTC', tx=Transaction(id=0x0000300000000039))
Bucket(name='demo-database/.snapshot/big_catalog_2024-06-14_08_35_25_UTC', tx=Transaction(id=0x0000300000000039))
Bucket(name='demo-database/.snapshot/big_catalog_2024-06-14_08_50_25_UTC', tx=Transaction(id=0x0000300000000039))


## Dimensions and Limits
Several limits are shared with other entities housed in VAST:
tables are analogous to files thus table count limits are shared with files
Snapshot count limits are shared across the system
Database count limit is shared with the platform view count

  
| Domain | Limit | Notes | 
| :--- | :----------- | :--- | 
| Max database count | 16,384 | This is limited by, and included in, the total number of views in the system | 
| Max table count | 1 billion per DBox | Limited by, and included in, the max file count. |
| Max row count | 2^48 (256 Trillion) | |
| Max column count | 31,936 |  |
| Max cell size | 126KB |  |
| Snapshot limit | 1019 | Same as for file systems |

**Note!**
The maximum column count will be increased to 64k in VAST 5.1

## Transaction Management

Transactions
VAST-DB is ACID compliant. All operations in VAST-DB are transactions, thus the boundaries of these transactions can be moved at will to cover very complex activities. The nature of how data is managed in the metastore makes transaction and transaction roll-back very low-latency. These mechanics are covered in the “Element Store” section.

We have been using transactions extensively during this tutorial, for example:

In [79]:
DATABASE_SCHEMA = 'this_should_never_get_created'

with session.transaction() as tx:

    bucket = tx.bucket(DATABASE_NAME)
    bucket.create_schema(DATABASE_SCHEMA, fail_if_exists=False)

    raise Exception("Forcing a failure inside the transaction - transaction should rollback")
    

rolling back txid=000030000000003a due to:
Traceback (most recent call last):
  File "/tmp/ipykernel_1103/47924204.py", line 8, in <module>
    raise Exception("Forcing a failure inside the transaction - transaction should rollback")
Exception: Forcing a failure inside the transaction - transaction should rollback


Exception: Forcing a failure inside the transaction - transaction should rollback

In the exception above, you should see `rolling back txid=...` indicating that the transaction was rolled back.

Now let's print out the schemas to verify that it wasn't created:


In [80]:
with session.transaction() as tx:

    bucket = tx.bucket(DATABASE_NAME)
    schemas = bucket.schemas()
    print(schemas)

[Schema(name='demo-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000003b))), Schema(name='python-sdk-schema', bucket=Bucket(name='demo-database', tx=Transaction(id=0x000030000000003b)))]
